In [ ]:
# ============================================================================
# KAGGLE SETUP - Run this cell FIRST
# ============================================================================

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# 1. INSTALL PYTORCH (CUDA 12.1) - Must use explicit index
# ----------------------------------------------------------------------------

print("Installing PyTorch with CUDA 12.1...")
result = subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1+cu121",
    "torchaudio==2.5.1+cu121",
    "torchvision==0.20.1+cu121"
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"❌ PyTorch install failed:\n{result.stderr[-3000:]}")
    raise RuntimeError("PyTorch installation failed")
print("✅ PyTorch installed")

# ----------------------------------------------------------------------------
# 2. INSTALL NUMPY FIRST - Locks ABI for binary wheels (insightface, onnxruntime, etc.)
# ----------------------------------------------------------------------------

print("Installing numpy==1.26.4 (must be first)...")
result = subprocess.run([
    sys.executable, "-m", "pip", "install", "numpy==1.26.4"
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"❌ NumPy install failed:\n{result.stderr[-3000:]}")
    raise RuntimeError("NumPy installation failed")
print("✅ NumPy installed")

# ----------------------------------------------------------------------------
# 3. INSTALL REMAINING REQUIREMENTS
# ----------------------------------------------------------------------------

requirements = """
# Audio Processing
# CRITICAL: PyPI package is 'pyannote-audio' (hyphen), NOT 'pyannote.audio' (dot)
pyannote-audio==3.3.2
faster-whisper==1.1.1
librosa==0.10.2.post1

# Transformers & NLP
transformers==4.44.0
huggingface-hub==0.25.2
tokenizers==0.19.1
accelerate==0.33.0
sentencepiece==0.2.0
hf-transfer==0.1.9

# Vision (InsightFace requires numpy<2.0)
insightface==0.7.3
opencv-python-headless==4.10.0.84
scikit-learn==1.5.1
scikit-image==0.24.0

# Face Analysis Dependencies
# onnxruntime-gpu only available on Linux; use onnxruntime on other platforms
onnxruntime-gpu==1.19.2

# Clustering & Metrics (numpy==1.26.4 CRITICAL for InsightFace compatibility)
scipy==1.13.1
numpy==1.26.4

# Data Processing
pandas==2.2.2
tqdm==4.66.4
pyyaml==6.0.1

# Utilities
ffmpeg-python==0.2.0
requests==2.32.3
python-dotenv==1.0.1

# LoRA / PEFT (QLoRA 4-bit quantization needs bitsandbytes)
peft==0.12.0
bitsandbytes==0.43.0

# Protobuf compatibility (required by Google Cloud libs, transformers)
protobuf==5.29.3

# FSSpec compatibility
fsspec==2025.3.0

# Rich for progress bars
rich==13.9.4

# Jupyter/Notebook (for Kaggle/Colab execution)
ipykernel==6.29.5
jupyter-client==8.6.2

# Download utilities
yt-dlp==2024.12.23

# Evaluation
jiwer==3.0.4
pyannote.metrics==3.2.1
"""

with open("requirements.txt", "w") as f:
    f.write(requirements.strip())

print("Installing remaining requirements...")
result = subprocess.run([
    sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"❌ Install failed:\n{result.stderr[-3000:]}")
    raise RuntimeError("Requirements installation failed")
else:
    print("✅ All requirements installed successfully")

In [ ]:
# ============================================================================
# 2. VERIFY NUMPY VERSION - CRITICAL FOR BINARY WHEEL COMPATIBILITY
# ============================================================================

import numpy as np
print(f"Current numpy version: {np.__version__}")

if not np.__version__.startswith("1.26"):
    print("⚠️ Wrong numpy version! Expected 1.26.x")
    print("Reinstalling numpy==1.26.4 with --force-reinstall...")
    import subprocess, sys
    result = subprocess.run([
        sys.executable, "-m", "pip", "install", "--force-reinstall", "numpy==1.26.4"
    ], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ NumPy reinstalled successfully")
        print("🔴 RESTART KERNEL NOW (Runtime → Restart session) then re-run this cell")
    else:
        print(f"❌ Failed: {result.stderr}")
else:
    print("✅ NumPy version OK (1.26.x)")
    print("✅ Binary wheel ABI compatibility confirmed")

In [ ]:
# ============================================================================
# 3. FIX NUMPY COMPATIBILITY (Runtime Patch for InsightFace)
# ============================================================================

import numpy as np
if not hasattr(np, "NaN"):
    np.NaN = np.nan
if not hasattr(np, "Inf"):
    np.Inf = np.inf
if not hasattr(np, "PINF"):
    np.PINF = np.inf
if not hasattr(np, "NINF"):
    np.NINF = -np.inf

print("✅ NumPy compatibility patch applied")

In [ ]:
def setup_hf_token():
    """Load HF token from Kaggle Secrets with fail-fast for gated models."""
    hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN") or ""
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            try:
                tok = user_secrets.get_secret(key)
                if tok and len(tok) > 20:
                    hf_token = tok
                    os.environ["HF_TOKEN"] = hf_token
                    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
                    print(f"✅ Hugging Face token loaded from Kaggle Secrets ({key})")
                    break
            except Exception:
                continue
    except Exception as e:
        print(f"⚠️ Could not load HF_TOKEN from secrets: {e}")
    if not hf_token or len(hf_token) < 20:
        print("⚠️ HF_TOKEN missing or too short — gated pyannote/speaker-diarization-3.1 will fail with 401")
        print("   Add HF_TOKEN (or HUGGING_FACE_HUB_TOKEN) in Kaggle Secrets (left sidebar 🔒) and restart kernel")
    else:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print(f"✅ HF token ready (length {len(hf_token)})")
    return hf_token


In [ ]:
# ============================================================================
# 5. VERIFY GPU AVAILABILITY
# ============================================================================

import torch
print("--- Hardware Verification ---")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"PyTorch CUDA version: {torch.version.cuda}")
else:
    print("⚠️ No GPU detected. Enable GPU in Kaggle session settings (Settings → Accelerator → GPU T4 x2)")

In [ ]:
# ============================================================================
# 6. CLONE REPOSITORY & SETUP PATHS
# ============================================================================

repo_url = "https://github.com/toufiq-dev/multimodal-speaker-indexing.git"
repo_dir = "/kaggle/working/multimodal-speaker-indexing"

if os.path.exists(repo_dir):
    import shutil
    shutil.rmtree(repo_dir)
    print("Removed existing repository")

print("Cloning repository...")
result = subprocess.run(["git", "clone", repo_url], capture_output=True, text=True)
if result.returncode != 0:
    print(f"Error cloning: {result.stderr}")
else:
    os.chdir(repo_dir)
    import importlib
    for mod in list(sys.modules.keys()):
        if mod == "config" or mod.startswith(("config.", "models", "engines")):
            del sys.modules[mod]
    importlib.invalidate_caches()
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    print(f"✅ Repository cloned to {repo_dir}")

In [ ]:
# ============================================================================
# 7. CREATE DATA DIRECTORIES
# ============================================================================

dirs = [
    "/kaggle/working/input",
    "/kaggle/working/registry",
    "/kaggle/working/output",
    "/kaggle/working/data/inputs",
    "/kaggle/working/data/registry",
    "/kaggle/working/data/output",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("✅ Data directories created")

In [ ]:
# ============================================================================
# 8. DOWNLOAD DATASET (Jamuna TV Rajniti Talk Show)
# ============================================================================

video_url = "https://youtu.be/qcMkD62HErQ"
output_path = "/kaggle/working/data/inputs/full_show.mp4"

if os.path.exists(output_path):
    print(f"✅ Video already exists: {output_path}")
else:
    print("Downloading video...")
    cmd = [
        "yt-dlp",
        "-f", "bestvideo[height<=720]+bestaudio/best[height<=720]",
        "--merge-output-format", "mp4",
        "-o", output_path,
        video_url
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error downloading: {result.stderr}")
    elif os.path.exists(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"✅ Video downloaded: {output_path} ({size_mb:.1f} MB)")
        
        # Get duration
        try:
            dur_cmd = ["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
                      "-of", "csv=p=0", output_path]
            dur_result = subprocess.run(dur_cmd, capture_output=True, text=True, check=True)
            duration = float(dur_result.stdout.strip())
            print(f"   Duration: {duration:.1f}s ({duration/60:.1f} min)")
        except:
            pass
    else:
        print("❌ Download failed - file not found")

In [ ]:
# ============================================================================
# 9. VERIFY ALL IMPORTS WORK
# ============================================================================

print("--- Verifying Imports ---")

modules_to_test = [
    ("config", "config"),
    ("models", "models"),
    ("engines.media", "engines.media"),
    ("engines.diarization", "engines.diarization"),
    ("engines.transcription", "engines.transcription"),
    ("engines.asr_lora", "engines.asr_lora"),
    ("engines.nlp", "engines.nlp"),
    ("engines.fusion", "engines.fusion"),
    # REMOVED: ("engines", "engines") - triggers lazy vision load
]

for name, module in modules_to_test:
    try:
        __import__(module)
        print(f"  ✅ {name}")
    except Exception as e:
        print(f"  ❌ {name}: {e}")

print("✅ All imports successful")

In [ ]:
# ============================================================================
# 10. QUICK ENGINE SMOKE TESTS
# ============================================================================

print("\n--- Quick Engine Smoke Tests ---")
import torch

# Test diarization (just load pipeline)
try:
    from engines.diarization import _load_pipeline
    pipeline = _load_pipeline()
    print(f"  ✅ diarization._load_pipeline: loaded")
    del pipeline
    torch.cuda.empty_cache()
except Exception as e:
    print(f"  ❌ diarization._load_pipeline: {e}")

# Test transcription model load
try:
    from engines.transcription import _load_model
    model = _load_model()
    print(f"  ✅ transcription._load_model: loaded")
    del model
    torch.cuda.empty_cache()
except Exception as e:
    print(f"  ❌ transcription._load_model: {e}")

# Test NER pipeline load
try:
    from engines.nlp import _load_ner_pipeline
    ner = _load_ner_pipeline()
    print(f"  ✅ nlp._load_ner_pipeline: loaded")
except Exception as e:
    print(f"  ❌ nlp._load_ner_pipeline: {e}")

# Test vision pipeline init (lazy load)
try:
    from engines.vision import run_vision_pipeline
    print(f"  ✅ vision.run_vision_pipeline: importable (lazy)")
except Exception as e:
    print(f"  ❌ vision.run_vision_pipeline: {e}")

# Test fusion
try:
    from engines.fusion import GatingFusion
    fusion = GatingFusion()
    print(f"  ✅ fusion.GatingFusion: instantiated")
except Exception as e:
    print(f"  ❌ fusion.GatingFusion: {e}")

# Test ASR LoRA
try:
    from engines.asr_lora import load_lora_whisper
    print(f"  ✅ asr_lora.load_lora_whisper: importable")
except Exception as e:
    print(f"  ❌ asr_lora.load_lora_whisper: {e}")

print("\n✅ All engine smoke tests complete")

In [ ]:
# ============================================================================
# PIPELINE EXECUTION - Run these cells in order
# ============================================================================

# ----------------------------------------------------------------------------
# CELL 1: Run audio-only pipeline (diarization + transcription)
# ----------------------------------------------------------------------------

import os
import torch
from engines.media import extract_audio
from engines.diarization import run_diarization
from engines.transcription import align_transcription_with_diarization

video_path = "/kaggle/working/data/inputs/full_show.mp4"

print("Extracting audio from video...")
audio_path = extract_audio(video_path)  # Use returned path
print(f"Using audio: {audio_path}")

print("Running speaker diarization...")
diarization = run_diarization(audio_path)
print(f"✅ Diarization complete: {len(diarization)} segments")
print(f"   Speakers found: {len(set(s.speaker_id for s in diarization))}")
torch.cuda.empty_cache()

print("Aligning transcription with diarization...")
transcribed = align_transcription_with_diarization(audio_path, diarization)
print(f"✅ Transcription complete: {len(transcribed)} segments")
torch.cuda.empty_cache()

for seg in transcribed[:5]:
    print(f"  [{seg.start:6.1f}s - {seg.end:6.1f}s] {seg.speaker_id}: {seg.text[:80]}...")

In [ ]:
# ============================================================================
# CELL 2: Run vision pipeline (face detection + recognition)
# ============================================================================

import torch
from engines.vision import run_vision_pipeline
from engines.media import extract_frames

video_path = "/kaggle/working/data/inputs/full_show.mp4"

print("Extracting frames...")
frame_paths = extract_frames(video_path, fps=1)
print(f"✅ Extracted {len(frame_paths)} frames")

print("Running vision pipeline (face detection + recognition)...")
faces = run_vision_pipeline(video_path, frame_paths=frame_paths)
print(f"✅ Vision complete: {len(faces)} face occurrences")

# Show unique faces found
unique_faces = set(f.resolved_face_id for f in faces)
print(f"   Unique faces: {sorted(unique_faces)}")

torch.cuda.empty_cache()

In [ ]:
# ============================================================================
# CELL 3: Extract speaker names from intro using NER
# ============================================================================

from engines.nlp import extract_speaker_names_from_intro

print("Extracting speaker names from intro...")
ordered_names = extract_speaker_names_from_intro(transcribed)
print(f"✅ Found names: {ordered_names}")

In [ ]:
# ============================================================================
# CELL 4: Run fusion to get final speaker-indexed segments
# ============================================================================

import torch
from engines.fusion import run_fusion_pipeline

print("Running fusion pipeline...")
final_segments = run_fusion_pipeline(diarization, transcribed, faces, ordered_names)
print(f"✅ Fusion complete: {len(final_segments)} final segments")

for seg in final_segments[:10]:
    print(f"  [{seg.start:6.1f}s - {seg.end:6.1f}s] {seg.speaker}: {seg.text[:80]}...")

torch.cuda.empty_cache()

In [ ]:
# ============================================================================
# CELL 5: Save results
# ============================================================================

import json
from pathlib import Path

output_dir = Path("/kaggle/working/output")
output_dir.mkdir(parents=True, exist_ok=True)

# Save JSON
data = [
    {
        "start": seg.start,
        "end": seg.end,
        "speaker": seg.speaker,
        "text": seg.text,
        "confidence": seg.confidence,
    }
    for seg in final_segments
]

with open(output_dir / "result.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# Save SRT
def fmt(t: float) -> str:
    h = int(t // 3600)
    m = int((t % 3600) // 60)
    s = int(t % 60)
    ms = int((t - int(t)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

with open(output_dir / "subtitles.srt", "w", encoding="utf-8") as f:
    for i, seg in enumerate(final_segments, 1):
        f.write(f"{i}\n")
        f.write(f"{fmt(seg.start)} --> {fmt(seg.end)}\n")
        f.write(f"{seg.speaker}: {seg.text}\n\n")

print(f"✅ Results saved to {output_dir}")
print(f"   - result.json")
print(f"   - subtitles.srt")

# ============================================================================
# CELL 6: END-TO-END VALIDATION
# ============================================================================

print("\n=== END-TO-END VALIDATION ===")
print(f"Total segments: {len(final_segments)}")
print(f"Total speakers: {len(set(s.speaker for s in final_segments))}")
print(f"Speakers: {sorted(set(s.speaker for s in final_segments))}")

# Verify output files exist and are valid
import json
result_file = output_dir / "result.json"
srt_file = output_dir / "subtitles.srt"

if result_file.exists():
    with open(result_file) as f:
        saved = json.load(f)
    print(f"✅ result.json: {len(saved)} segments saved")
else:
    print("❌ result.json: NOT FOUND")

if srt_file.exists():
    with open(srt_file) as f:
        srt_content = f.read()
    print(f"✅ subtitles.srt: {len(srt_content)} chars, {srt_content.count('-->')} entries")
else:
    print("❌ subtitles.srt: NOT FOUND")

# Print first 5 final segments for verification
print("\nFirst 5 final segments:")
for seg in final_segments[:5]:
    print(f"  [{seg.start:6.1f}s - {seg.end:6.1f}s] {seg.speaker} (conf={seg.confidence:.2f}): {seg.text[:80]}...")

In [ ]:
# ============================================================================
# ALTERNATIVE: Run full pipeline in one call
# ============================================================================

# from main import run_pipeline

# final_segments = run_pipeline(
#     video_path="/kaggle/working/data/inputs/full_show.mp4",
#     registry_dir="/kaggle/working/data/registry",  # Optional: add face images here
#     output_dir="/kaggle/working/output",
#     use_lora=False,  # Set True if you have a LoRA adapter
#     lora_path=None
# )
# print(f"Pipeline complete: {len(final_segments)} segments")